In [13]:
!pip install pytorch-lightning

In [14]:
from IPython.display import display

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader, random_split


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCH = 20

df = pd.read_csv('./dataset/data_cleaning.csv')
X, y = df.drop(columns='cardio', inplace=False), df['cardio']

display(X.head())

,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,ag_stage,bmi,age_years,age_groups
0,2,168,62.0,110,80,1,1,0,0,1,-1,22.0,50,2
1,1,156,85.0,140,90,3,1,0,0,1,1,35.0,55,2
2,1,165,64.0,130,70,3,1,0,0,0,-1,24.0,51,2
3,2,169,82.0,150,100,1,1,0,0,1,-1,29.0,48,1
4,1,156,56.0,100,60,1,1,0,0,0,-1,23.0,47,1


In [15]:
scaler = StandardScaler()
scaled_X = scaler.fit_transform(X)

scaled_X

array([[ 1.36403989,  0.45217522, -0.8537876 , ..., -1.04528311,
        -0.4201051 ,  0.66376649],
       [-0.73311639, -1.05776727,  0.75816977, ...,  1.43570517,
         0.31880809,  0.66376649],
       [-0.73311639,  0.0746896 , -0.71361739, ..., -0.66359261,
        -0.27232246,  0.66376649],
       ...,
       [ 1.36403989,  2.33960333,  2.15987184, ...,  0.67232416,
        -0.12453983,  0.66376649],
       [-0.73311639, -0.17696748, -0.15293657, ..., -0.09105685,
         1.20550393,  0.66376649],
       [-0.73311639,  0.7038323 , -0.15293657, ..., -0.47274735,
         0.46659073,  0.66376649]], shape=(69527, 14))

**Критерий Кайзера** (правило «собственное значение > 1») — это статистический метод в факторном анализе и методе главных компонент (PCA), используемый для определения оптимального количества сохраняемых факторов. Согласно критерию, следует оставлять только те факторы (компоненты), чьи собственные значения (eigenvalues) больше 1.

In [16]:
pca = PCA()
X_pca = pca.fit(scaled_X)

variance = X_pca.explained_variance_
cnt_component = variance[variance > 1].shape[0]

display(variance)
display(cnt_component)

array([2.91060763, 2.02914845, 1.61717642, 1.45517394, 1.28506544,
       1.08098511, 0.98791338, 0.65229114, 0.53982489, 0.52771527,
       0.47638302, 0.25450954, 0.17677353, 0.0066336 ])

6

In [17]:
pca = PCA(n_components=cnt_component)
X_pca = pca.fit_transform(scaled_X)

X_pca.shape, scaled_X.shape

((69527, 6), (69527, 14))

In [18]:
tensor_X = torch.from_numpy(X_pca).to(DEVICE)
tensor_y = torch.from_numpy(y.values).to(DEVICE)


print(DEVICE)

cpu


In [20]:
class MyData(Dataset):
    def __init__(self):
        self.x = tensor_X
        self.y = tensor_y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]


data = MyData()
train_data, val_data, test_data = random_split(data, [.6, .2, .2])

train_loader = DataLoader(train_data, shuffle=True, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_data, shuffle=False, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, shuffle=False, batch_size=BATCH_SIZE)